In [2]:
import cv2
import mediapipe as mp

# -----------------------------------
# GitHub Source Link
# -----------------------------------
github_link = "https://github.com/ajanthadevi2012"

# -----------------------------------
# MediaPipe Pose Setup
# -----------------------------------
pose_api = mp.solutions.pose
draw = mp.solutions.drawing_utils

pose = pose_api.Pose(
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

# -----------------------------------
# Open Webcam
# -----------------------------------
cap = cv2.VideoCapture(0)

# Get camera properties
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

if fps == 0:
    fps = 30

# -----------------------------------
# Output Video
# -----------------------------------
output_path = "hands_up_down_counter_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

# -----------------------------------
# Counter Variables
# -----------------------------------
count = 0
hands_up = False

# Thresholds
up_thres = 0.35
down_thres = 0.65

# -----------------------------------
# Main Loop
# -----------------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Mirror view
    frame = cv2.flip(frame, 1)

    # Convert BGR to RGB
    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # Process pose
    result = pose.process(rgb)

    status = "No Pose Detected"

    if result.pose_landmarks:

        lm = result.pose_landmarks.landmark

        # -----------------------------------
        # Get Left and Right Wrist Positions
        # -----------------------------------
        left_wrist = lm[
            pose_api.PoseLandmark.LEFT_WRIST
        ]

        right_wrist = lm[
            pose_api.PoseLandmark.RIGHT_WRIST
        ]

        # Average position of both hands
        hands_y = (
            left_wrist.y +
            right_wrist.y
        ) / 2

        # -----------------------------------
        # Detect Hands UP
        # -----------------------------------
        if hands_y < up_thres:
            hands_up = True

        # -----------------------------------
        # Detect Hands DOWN
        # -----------------------------------
        if hands_up and hands_y > down_thres:

            count += 1
            hands_up = False

        # Status
        status = "UP" if hands_up else "DOWN"

        # Draw pose landmarks
        draw.draw_landmarks(
            frame,
            result.pose_landmarks,
            pose_api.POSE_CONNECTIONS
        )

    # -----------------------------------
    # Display Counter
    # -----------------------------------
    cv2.putText(
        frame,
        f"Folds: {count}",
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,0, 0),
        2
    )

    # -----------------------------------
    # Display Status
    # -----------------------------------
    cv2.putText(
        frame,
        f"Status: {status}",
        (20, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255,0, 0),
        2
    )

    # -----------------------------------
    # Display GitHub Link - Top Right
    # -----------------------------------
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.45
    thickness = 1

    (text_width, text_height), _ = cv2.getTextSize(
        github_link,
        font,
        font_scale,
        thickness
    )

    x = width - text_width - 10
    y = 25

    cv2.putText(
        frame,
        github_link,
        (x, y),
        font,
        font_scale,
        (255, 255, 255),
        thickness
    )

    # -----------------------------------
    # Save Processed Frame
    # -----------------------------------
    out.write(frame)

    # Display Output
    cv2.imshow(
        "Hands UP/DOWN Counter",
        frame
    )

    # Press Q to Stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# -----------------------------------
# Release Resources
# -----------------------------------
cap.release()
out.release()
pose.close()
cv2.destroyAllWindows()

print("Video saved as:", output_path)

Video saved as: hands_up_down_counter_output.mp4
